# Keithley 3706A-S / 3723-ST + Keithley 2400 Continuity Scanner

Notebook-first UI for linked switch scanning. Select the VS Code Notebook kernel from the AI conda environment: `C:\\Users\\liy56\\.conda\\envs\\AI\\python.exe`.

Safety sequence for every pair: 2400 output off, 3706 open all, close selected relay channels, wait, 2400 output on, measure, 2400 output off, 3706 open all.

3723-ST is a multiplexer terminal board, not a matrix. This setup is wired on the slot 4 MUX outputs, so the default channel ranges are MUX1 `4001-4030` and MUX2 `4031-4060`.


In [ ]:
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

try:
    from ipyfilechooser import FileChooser
except ImportError:
    FileChooser = None

from keithley3706 import Keithley3706A, probe_visa_resources
from keithley2400 import Keithley2400
from scan_logic import generate_crossbar_pairs, parse_manual_pairs, run_continuity_scan


In [ ]:
switch = Keithley3706A()
meter = Keithley2400()
latest_results = pd.DataFrame()
detected_resources = []

status_out = widgets.Output()
results_out = widgets.Output()

def log_status(message):
    with status_out:
        print(message)

def resource_options():
    options = [("Manual / empty", "")]
    for item in detected_resources:
        label = item["resource"]
        if item.get("identity"):
            label = f"{label} - {item['identity']}"
        elif item.get("error"):
            label = f"{label} - ERROR: {item['error']}"
        options.append((label, item["resource"]))
    return options

def selected_resource(dropdown, manual_text):
    manual = manual_text.value.strip()
    return manual or dropdown.value or None


In [ ]:
detect_button = widgets.Button(description="Detect instruments", button_style="info")
switch_dropdown = widgets.Dropdown(description="3706", options=resource_options())
meter_dropdown = widgets.Dropdown(description="2400", options=resource_options())
switch_manual = widgets.Text(description="3706 manual", placeholder="GPIB0::18::INSTR")
meter_manual = widgets.Text(description="2400 manual", placeholder="GPIB0::24::INSTR")
connect_switch_button = widgets.Button(description="Connect 3706", button_style="success")
connect_meter_button = widgets.Button(description="Connect 2400", button_style="success")
disconnect_button = widgets.Button(description="Disconnect all", button_style="warning")
switch_idn = widgets.Textarea(description="3706 IDN", disabled=True, layout=widgets.Layout(width="95%", height="50px"))
meter_idn = widgets.Textarea(description="2400 IDN", disabled=True, layout=widgets.Layout(width="95%", height="50px"))

def on_detect(_):
    global detected_resources
    status_out.clear_output()
    try:
        detected_resources = probe_visa_resources()
        options = resource_options()
        switch_dropdown.options = options
        meter_dropdown.options = options
        for item in detected_resources:
            identity = item.get("identity", "")
            if "3706" in identity:
                switch_dropdown.value = item["resource"]
            if "2400" in identity:
                meter_dropdown.value = item["resource"]
        log_status(f"Detected {len(detected_resources)} VISA resources")
    except Exception as exc:
        log_status(f"Detect failed: {exc}")

def on_connect_switch(_):
    try:
        identity = switch.connect(resource=selected_resource(switch_dropdown, switch_manual))
        switch_idn.value = identity
        log_status("3706 connected")
    except Exception as exc:
        log_status(f"3706 connect failed: {exc}")

def on_connect_meter(_):
    try:
        identity = meter.connect(resource=selected_resource(meter_dropdown, meter_manual))
        meter_idn.value = identity
        log_status("2400 connected and output is off")
    except Exception as exc:
        log_status(f"2400 connect failed: {exc}")

def on_disconnect(_):
    try:
        meter.disconnect()
        switch.disconnect()
        log_status("Disconnected")
    except Exception as exc:
        log_status(f"Disconnect issue: {exc}")

detect_button.on_click(on_detect)
connect_switch_button.on_click(on_connect_switch)
connect_meter_button.on_click(on_connect_meter)
disconnect_button.on_click(on_disconnect)

display(widgets.VBox([
    widgets.HBox([detect_button, connect_switch_button, connect_meter_button, disconnect_button]),
    widgets.HBox([switch_dropdown, meter_dropdown]),
    widgets.HBox([switch_manual, meter_manual]),
    switch_idn,
    meter_idn,
    status_out,
]))


In [ ]:
manual_channels = widgets.Text(description="Channels", value="4001", placeholder="1001,1002 or 1001:1008", layout=widgets.Layout(width="60%"))
close_button = widgets.Button(description="Close", button_style="warning")
open_button = widgets.Button(description="Open", button_style="info")
open_all_button = widgets.Button(description="Open All", button_style="warning")
refresh_closed_button = widgets.Button(description="Refresh Closed", button_style="")
safe_state_button = widgets.Button(description="Emergency Off", button_style="danger")
closed_channels = widgets.Textarea(description="Closed", disabled=True, layout=widgets.Layout(width="95%", height="60px"))
manual_note = widgets.HTML(value="<b>Manual relay control:</b> confirm the 2400 output is off before switching relays. Use Emergency Off to turn 2400 output off and open all 3706 channels.")

def refresh_closed_channels():
    try:
        closed_channels.value = switch.get_closed_channels()
    except Exception as exc:
        log_status(f"Refresh closed channels failed: {exc}")

def on_manual_close(_):
    try:
        closed_channels.value = switch.close_channels(manual_channels.value)
        log_status(f"Closed {manual_channels.value}")
    except Exception as exc:
        log_status(f"Close failed: {exc}")

def on_manual_open(_):
    try:
        closed_channels.value = switch.open_channels(manual_channels.value)
        log_status(f"Opened {manual_channels.value}")
    except Exception as exc:
        log_status(f"Open failed: {exc}")

def on_manual_open_all(_):
    try:
        closed_channels.value = switch.open_all()
        log_status("Opened all 3706 channels")
    except Exception as exc:
        log_status(f"Open all failed: {exc}")

def on_refresh_closed(_):
    refresh_closed_channels()

def on_safe_state(_):
    errors = []
    try:
        meter.output_off()
    except Exception as exc:
        errors.append(f"2400 output off failed: {exc}")
    try:
        closed_channels.value = switch.open_all()
    except Exception as exc:
        errors.append(f"3706 open all failed: {exc}")
    if errors:
        log_status("Emergency Off completed with issues: " + "; ".join(errors))
    else:
        log_status("Emergency Off complete: 2400 output off, 3706 open all")

close_button.on_click(on_manual_close)
open_button.on_click(on_manual_open)
open_all_button.on_click(on_manual_open_all)
refresh_closed_button.on_click(on_refresh_closed)
safe_state_button.on_click(on_safe_state)

display(widgets.VBox([
    manual_note,
    widgets.HBox([manual_channels, close_button, open_button, open_all_button, refresh_closed_button, safe_state_button]),
    closed_channels,
]))


In [ ]:
scan_mode = widgets.ToggleButtons(description="Mode", options=[("A x B", "crossbar"), ("Manual", "manual")])
slot_note = widgets.HTML(value="<b>Slot 4 wiring:</b> 2400 is connected to the slot 4 MUX outputs. Defaults use MUX1 4001-4030 and MUX2 4031-4060.")
a_start = widgets.IntText(description="A start", value=4001)
a_end = widgets.IntText(description="A end", value=4030)
b_start = widgets.IntText(description="B start", value=4031)
b_end = widgets.IntText(description="B end", value=4060)
manual_pairs = widgets.Textarea(
    description="Pairs",
    value="4001,4031\n4002,4032",
    layout=widgets.Layout(width="95%", height="100px"),
)
source_mode = widgets.ToggleButtons(description="Source", options=[("Voltage", "voltage"), ("Current", "current")])
source_level = widgets.FloatText(description="Level", value=0.1)
compliance = widgets.FloatText(description="Compliance", value=0.01)
settle_time = widgets.FloatText(description="Settle s", value=0.05)
threshold = widgets.FloatText(description="Ohm limit", value=10.0)
stop_on_error = widgets.Checkbox(description="Stop on first ERROR", value=True, indent=False)
abort_note = widgets.HTML(value="To abort immediately while a scan is running, use Jupyter <b>Kernel > Interrupt Kernel</b>. The current pair still attempts 2400 output off and 3706 open all in cleanup.")
run_button = widgets.Button(description="Run scan", button_style="danger")
csv_path = widgets.Text(description="CSV path", value="continuity_results.csv", layout=widgets.Layout(width="70%"))
save_button = widgets.Button(description="Save CSV", button_style="primary")
file_chooser = FileChooser(".") if FileChooser is not None else None
if file_chooser is not None:
    file_chooser.title = "Browse CSV save location"
    file_chooser.default_filename = "continuity_results.csv"
    file_chooser.show_only_dirs = False
    file_chooser.filter_pattern = ["*.csv"]
    def on_file_chosen(chooser):
        selected = chooser.selected
        if selected:
            csv_path.value = selected
    file_chooser.register_callback(on_file_chosen)
else:
    file_chooser = widgets.HTML(value="Install ipyfilechooser to enable Browse: <code>python -m pip install ipyfilechooser</code>")

def build_pairs():
    if scan_mode.value == "manual":
        return parse_manual_pairs(manual_pairs.value)
    return generate_crossbar_pairs(a_start.value, a_end.value, b_start.value, b_end.value)

def configure_meter():
    if source_mode.value == "voltage":
        meter.configure_source_voltage(source_level.value, compliance.value)
    else:
        meter.configure_source_current(source_level.value, compliance.value)

def on_run_scan(_):
    global latest_results
    results_out.clear_output()
    try:
        pairs = build_pairs()
        configure_meter()
        rows = run_continuity_scan(
            switch=switch,
            meter=meter,
            pairs=pairs,
            source_mode=source_mode.value,
            source_level=source_level.value,
            compliance=compliance.value,
            settle_time_s=settle_time.value,
            resistance_threshold=threshold.value,
            switch_idn=switch_idn.value,
            meter_idn=meter_idn.value,
            stop_on_error=stop_on_error.value,
        )
        latest_results = pd.DataFrame(rows)
        with results_out:
            display(latest_results)
    except KeyboardInterrupt:
        try:
            meter.output_off()
        except Exception:
            pass
        try:
            switch.open_all()
        except Exception:
            pass
        with results_out:
            print("Scan interrupted. Attempted 2400 output off and 3706 open all.")
    except Exception as exc:
        with results_out:
            print(f"Scan failed before first pair: {exc}")

def on_save_csv(_):
    if latest_results.empty:
        log_status("No results to save")
        return
    path = Path(csv_path.value).expanduser()
    latest_results.to_csv(path, index=False)
    log_status(f"Saved {len(latest_results)} rows to {path}")

run_button.on_click(on_run_scan)
save_button.on_click(on_save_csv)

display(widgets.VBox([
    slot_note,
    scan_mode,
    widgets.HBox([a_start, a_end, b_start, b_end]),
    manual_pairs,
    widgets.HBox([source_mode, source_level, compliance]),
    widgets.HBox([settle_time, threshold, stop_on_error]),
    abort_note,
    file_chooser,
    widgets.HBox([run_button, csv_path, save_button]),
    results_out,
]))


In [ ]:
#cleaning queue
for i in range(50):
    try:
        err = switch.query_expression("errorqueue.next()")
        print(i, err)
        if "No error" in err or err.strip().startswith("0"):
            break
    except Exception as exc:
        print("read errorqueue failed:", exc)
        break

In [ ]:
#reset
switch.write_raw("channel.openall()")